In [ ]:
# !pip install ipython==8.1.0

In [ ]:
# # Comment the following if you are running your code locally
# # This mounts your Google Drive to the Colab VM.
# from google.colab import drive
# drive.mount('/content/drive')
# drive_folder = '/content/drive/MyDrive'
# notebook_folder = drive_folder + '/project'
# %cd {notebook_folder}

# You don't need to run unless the csv files are missing
# %cd ./cs682/data
# !bash prepare.sh
# %cd ../../

In [ ]:
%load_ext autoreload
%autoreload 2

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from transformers import get_linear_schedule_with_warmup
import json

from cs682.models.teacher import BERTTeacher
from cs682.data.loader import IMDBDataset, YelpDataset, AmazonDataset
from cs682.evaluator import evaluate

In [ ]:
class TeacherTrainConfig:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

    def __repr__(self):
        return f"TeacherTrainConfig({self.__dict__})"


def check_accuracy(
    model: BERTTeacher, loader: DataLoader, device, max_batches: int = None
):
    model.eval()
    correct = 0
    total = 0
    use_amp = device.type == "cuda"
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["labels"].to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(input_ids=input_ids, attention_mask=attention_mask)
            predicted = torch.argmax(out["logits"], dim=1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
    return correct / total


def train(
    config: TeacherTrainConfig, train_loader: DataLoader, validation_loader: DataLoader
):
    print(config)

    criterion = config.criterion
    optimizer = config.optimizer
    scheduler = config.scheduler
    model = config.model
    max_grad_norm = config.max_grad_norm

    epochs = config.epochs
    log_every_k = config.log_every_k
    accuracy_every_k = config.accuracy_every_k

    device = config.device
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)

    model.to(device)

    losses = []
    train_accuracies = []
    validation_accuracies = []

    step = 0
    for epoch in range(epochs):
        model.train()

        for i, batch in enumerate(train_loader):
            step += 1

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["labels"].to(device)

            optimizer.zero_grad()

            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(out["logits"], targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            losses.append(loss.item())

            if step % log_every_k == 0:
                lr = scheduler.get_last_lr()[0]
                print(
                    f"Epoch {epoch + 1}/{epochs}, Step {step}, Batch {i}/{len(train_loader)}, Loss: {losses[-1]:.4f}, LR: {lr:.2e}"
                )

            if step % accuracy_every_k == 0:
                max_batches = max(1, len(train_loader) // 10)
                train_acc = check_accuracy(
                    model, train_loader, device, max_batches=max_batches
                )
                val_acc = check_accuracy(
                    model, validation_loader, device, max_batches=max_batches
                )
                train_accuracies.append(train_acc)
                validation_accuracies.append(val_acc)
                print(f"  Train Acc: {train_acc:.4f}  |  Val Acc: {val_acc:.4f}")
                model.train()

    return model, losses, train_accuracies, validation_accuracies


def make_collate_fn(tokenizer, max_length):
    def collate_fn(batch):
        texts, labels = zip(*batch)
        enc = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc

    return collate_fn

In [ ]:
dataset_name = "imdb" # imdb, yelp, amazon
is_two_classes = True # only applied for Amazon or Yelp, leave as True for IMDB
epochs = 3
learning_rate = 2e-5
weight_decay = 1e-4
warmup_ratio = 0.06
max_grad_norm = 1.0
validation_pct = 0.2
batch_size = 32
num_workers = 4
log_every_k = 50
accuracy_every_k = 200
max_length = 128
dropout = 0.1
model_save_path = f"./models/teacher_{dataset_name}_{is_two_classes}.pt"
train_save_path = f"./models/teacher_{dataset_name}_{is_two_classes}.json"
mapped_layer_indices = [4, 8] # you can leave this as is, during distillation it will change

# change from args to regular variables
num_classes = 2 if is_two_classes else 5

if dataset_name == "imdb":
    dataset = IMDBDataset()
    num_classes = 2
elif dataset_name == "yelp":
    dataset = YelpDataset(is_two_classes=is_two_classes)
elif dataset_name == "amazon":
    dataset = AmazonDataset(is_two_classes=is_two_classes)
else:
    raise ValueError()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, tokenizer = BERTTeacher.from_pretrained(
    num_classes=num_classes,
    mapped_layer_indices=mapped_layer_indices,
    dropout=dropout,
)

val_size = int(len(dataset) * validation_pct)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

collate_fn = make_collate_fn(tokenizer, max_length)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
total_steps = len(train_loader) * epochs
warmup_steps = int(total_steps * warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

train_config = TeacherTrainConfig(
    model=model,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
    log_every_k=log_every_k,
    accuracy_every_k=accuracy_every_k,
    max_grad_norm=max_grad_norm,
    device=device,
)

model, losses, train_accs, val_accs = train(train_config, train_loader, val_loader)

with open(train_save_path, "w") as f:
    obj = {
        "losses": losses,
        "train_accs": train_accs,
        "validation_accs": val_accs
    }

    json.dump(obj, f, indent=2)

train_config_dict = {
    "dataset": dataset,
    "is_two_classes": is_two_classes,
    "epochs": epochs,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "warmup_ratio": warmup_ratio,
    "max_grad_norm": max_grad_norm,
    "validation_pct": validation_pct,
    "batch_size": batch_size,
    "num_workers": num_workers,
    "log_every_k": log_every_k,
    "accuracy_every_k": accuracy_every_k,
    "max_length": max_length,
    "dropout": dropout,
    "mapped_layer_indices": mapped_layer_indices,
}

torch.save({
    "model_state_dict": model.state_dict(),
    "args": train_config_dict,
}, model_save_path)
print(f"\nModel saved to {model_save_path}")


In [ ]:
num_classes = 2 if is_two_classes else 5
if dataset_name == "imdb":
    dataset = IMDBDataset(split="test")
    num_classes = 2
elif dataset_name == "yelp":
    dataset = YelpDataset(split="test", is_two_classes=is_two_classes)
elif dataset_name == "amazon":
    dataset = AmazonDataset(split="test", is_two_classes=is_two_classes)
else:
    raise ValueError()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

evaluate(model, loader, num_classes, device)